# Shared Preprocessing Notebook
## DataCo Supply Chain Dataset — Version 5

**Team purpose:** use one documented preprocessing policy for the four prediction tasks: fraud classification, late-delivery classification, sales regression, and benefit-per-order regression.

**Shared split settings:** 80% training / 20% testing, `random_state=42`; stratify classification splits.

**Important:** The supplied `DataCoSupplyChainDataset.csv` contains 180,519 rows and 53 columns. It includes `Late_delivery_risk`, `Sales`, and `Benefit per order`, but **does not contain a fraud-label column**. Fraud model results supplied by a teammate are recorded below as reported baseline results; they are not reproduced from this CSV. A genuine fraud-labelled dataset or a verified fraud-label file is required to train/evaluate that task.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20

# This notebook is stored in notebooks/, while the raw dataset should be in the repository root.
DATA_PATH = Path("../DataCoSupplyChainDataset.csv")
RESULTS_DIR = Path("../results")
SPLITS_DIR = RESULTS_DIR / "splits"

In [ ]:
# Load the original V5 dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Place DataCoSupplyChainDataset.csv in the repository root "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH, encoding="latin1", low_memory=False)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

In [ ]:
# Dataset exploration
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nDuplicate rows:", df.duplicated().sum())

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)
display(missing[missing["missing_count"] > 0])

display(df.describe(include="all").T)

## 1. Prediction targets and availability

The target names below use the **actual column names in the uploaded CSV**.

| Task | Dataset target column | Type | Availability |
|---|---|---|---|
| Fraud | `fraud` (expected label) | Classification | Not present in supplied CSV |
| Late delivery | `Late_delivery_risk` | Classification | Present |
| Sales | `Sales` | Regression | Present |
| Benefit per order | `Benefit per order` | Regression | Present |

`Type` is a payment/transaction category, not a fraud label. It must not be renamed or treated as the fraud target.

The teammate-reported classifier results are retained as a baseline reference below. They are not evidence that fraud labels exist in this DataCo CSV.

In [ ]:
# Verify the target columns and inspect target distributions
TARGET_COLUMNS = {
    "fraud": "fraud",
    "late_delivery": "Late_delivery_risk",
    "sales": "Sales",
    "benefit_per_order": "Benefit per order",
}

for task, target_col in TARGET_COLUMNS.items():
    if target_col not in df.columns:
        print(f"{task}: target '{target_col}' NOT FOUND in this dataset")
        continue

    print(f"\\n{task} — {target_col}")
    print("Missing target values:", int(df[target_col].isna().sum()))
    display(df[target_col].value_counts(dropna=False).head(20))

## 2. Teammate-reported classification baseline

The project information supplied by the team states that nine classifiers were compared on fraud and late delivery, and that Decision Tree performed best on both:

- Fraud: F1 = **0.60**
- Late delivery: F1 = **0.88**
- Agreed split: 80/20, stratified, `random_state=42`

These are **reported results**, not independently recalculated here. The source file is expected at `results/classification_results.csv` after the teammate's PR is merged or its file is downloaded into this repository.

In [ ]:
# Load the teammate's full classifier-results CSV if it has been placed in results/.
classification_results_path = RESULTS_DIR / "classification_results.csv"

if classification_results_path.exists():
    classification_results = pd.read_csv(classification_results_path)
    print("Loaded teammate classification results:")
    print("Shape:", classification_results.shape)
    print("Columns:", classification_results.columns.tolist())
    display(classification_results)
else:
    classification_results = None
    print(
        "results/classification_results.csv not found. "
        "The teammate-reported Decision Tree F1 scores are shown as reference only."
    )

reported_baseline = pd.DataFrame([
    {"task": "fraud", "best_model": "Decision Tree", "reported_f1": 0.60,
     "source": "Team-provided result; awaiting classification_results.csv"},
    {"task": "late_delivery", "best_model": "Decision Tree", "reported_f1": 0.88,
     "source": "Team-provided result; awaiting classification_results.csv"},
])
display(reported_baseline)

## 3. Leakage prevention

The exclusions below follow the team's instructions and the actual column names in this dataset. Leakage exclusions are **task-specific**: the feature list for one target should not automatically be reused for another.

- **Late-delivery classification:** remove `Order Status`, `Delivery Status`, `Days for shipping (real)`, and `shipping date (DateOrders)`. These reveal delivery outcomes or information only known after the event.
- **Sales regression:** remove the target `Sales` and direct sales proxies including `Order Item Total`, `Product Price`, `Order Item Product Price`, and `Sales per customer`.
- **Benefit-per-order regression:** remove the target `Benefit per order` and closely related outcome-derived fields such as `Order Profit Per Order` and `Order Item Profit Ratio`. The team should confirm this target definition and exclusion policy together.
- **Fraud classification:** cannot be prepared until the genuine fraud label and its source are supplied. Do not use `Type` as a substitute.

The notebook also excludes direct identifiers and personal/customer information from model features. Do not use customer passwords, email addresses, names, or street addresses as predictors.

In [ ]:
# Shared exclusions
IDENTIFIER_AND_PERSONAL_COLUMNS = [
    "Customer Email", "Customer Password", "Customer Fname", "Customer Lname",
    "Customer Street", "Customer Id", "Order Customer Id", "Order Id",
    "Order Item Id", "Customer Zipcode", "Order Zipcode",
    "Product Description", "Product Image",
]

TASK_LEAKAGE_COLUMNS = {
    "fraud": [],
    "late_delivery": [
        "Order Status",
        "Delivery Status",
        "Days for shipping (real)",
        "shipping date (DateOrders)",
    ],
    "sales": [
        "Order Item Total",
        "Product Price",
        "Order Item Product Price",
        "Sales per customer",
    ],
    "benefit_per_order": [
        "Order Profit Per Order",
        "Order Item Profit Ratio",
    ],
}

def get_task_features(data, task, target_column):
    if target_column not in data.columns:
        raise ValueError(
            f"Cannot prepare {task}: target '{target_column}' is not in the supplied dataset."
        )

    excluded = set(
        IDENTIFIER_AND_PERSONAL_COLUMNS
        + TASK_LEAKAGE_COLUMNS[task]
        + [target_column]
    )
    feature_columns = [col for col in data.columns if col not in excluded]
    return data[feature_columns].copy(), data[target_column].copy(), feature_columns

## 4. Reproducible train/test split and preprocessing

Use the same split settings for every model comparing the **same task**:

- `test_size=0.20`
- `random_state=42`
- `stratify=y` for classification; no stratification for regression.

Each task has its own target and may have different rows with missing target values. Therefore, persist the train/test row indices per task and share those indices with teammates. This ensures every model for a given task uses the exact same rows.

Imputation, one-hot encoding, and scaling are fitted on the training features only. The test set is transformed using the already-fitted preprocessing pipeline.

In [ ]:
def make_preprocessor(X_train):
    numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])

def prepare_task(data, task, target_column, task_type):
    if target_column not in data.columns:
        print(f"SKIPPED {task}: target '{target_column}' is unavailable.")
        return None

    task_data = data.dropna(subset=[target_column]).copy()
    X, y, feature_columns = get_task_features(task_data, task, target_column)

    stratify = y if task_type == "classification" else None

    # Fail clearly rather than silently creating an invalid stratified split.
    if task_type == "classification" and y.value_counts().min() < 2:
        raise ValueError(
            f"{task}: at least one class has fewer than 2 rows; "
            "stratified 80/20 splitting is not possible."
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify,
    )

    preprocessor = make_preprocessor(X_train)
    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    # Save row indices as JSON so teammates can reproduce the exact same split.
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    split_record = {
        "task": task,
        "target_column": target_column,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "task_type": task_type,
        "train_index": X_train.index.tolist(),
        "test_index": X_test.index.tolist(),
    }
    with open(SPLITS_DIR / f"{task}_split_indices.json", "w") as f:
        json.dump(split_record, f)

    print(
        f"{task}: train={len(X_train):,}, test={len(X_test):,}, "
        f"raw_features={len(feature_columns)}, "
        f"processed_features={X_train_processed.shape[1]}"
    )

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "X_train_processed": X_train_processed,
        "X_test_processed": X_test_processed,
        "preprocessor": preprocessor,
        "feature_columns": feature_columns,
    }

In [ ]:
# Prepare all tasks supported by this CSV.
# Fraud remains listed but is skipped because the genuine label is absent.
task_splits = {}

TASK_SPECS = [
    ("fraud", "fraud", "classification"),
    ("late_delivery", "Late_delivery_risk", "classification"),
    ("sales", "Sales", "regression"),
    ("benefit_per_order", "Benefit per order", "regression"),
]

for task, target_col, task_type in TASK_SPECS:
    result = prepare_task(df, task, target_col, task_type)
    if result is not None:
        task_splits[task] = result

print("\\nPrepared tasks:", list(task_splits.keys()))
print("Fraud is pending a genuine labelled fraud dataset/column.")

## 5. Shared team checklist

- [ ] All members use DataCoSupplyChainDataset.csv V5 for the DataCo tasks.
- [ ] All members use 80/20, `random_state=42`, and stratification for classification.
- [ ] For each task, all models use the same saved train/test row-index JSON file.
- [ ] Leakage exclusions above are applied consistently.
- [ ] Preprocessors are fitted only on training data.
- [ ] A genuine fraud label source is obtained and documented before fraud neural-network training.
- [ ] The teammate's `results/classification_results.csv` is added from the PR and retained as the nine-classifier baseline.
- [ ] Neural-network results are exported to CSV using the same task/model/metric naming convention where possible.
- [ ] Regression comparisons include MAE and RMSE.
- [ ] Raw dataset files are not committed to GitHub.

**Reported baseline to retain:** Decision Tree — fraud F1 0.60; late-delivery F1 0.88. These scores should be replaced or supplemented with the exact rows from `classification_results.csv` once that file is available.